## MacroSense | Notebook 1: Data Collection

## Author: Aremu Gideon Marvelous

## Date: May 2026

## Description: Connects to FRED API and downloads

## all required economic data series

In [1]:
# Import all required libraries
from fredapi import Fred       # connects to FRED database
from dotenv import load_dotenv # loads our secret API key safely
import pandas as pd            # handles our data tables
import os                      # helps us work with files and folders

print(" All libraries imported successfully")

 All libraries imported successfully


In [2]:
# Load API key safely from .env file
load_dotenv()
api_key = os.getenv('FRED_API_KEY')

# Connect to FRED using our API key
fred = Fred(api_key=api_key)

# Test the connection by downloading one small series
test = fred.get_series('GDPC1')

print(" API key loaded successfully")
print(" Connected to FRED successfully")
print(f" Test series retrieved: {len(test)} observations")
print(f" Date range: {test.index[0].date()} to {test.index[-1].date()}")

 API key loaded successfully
 Connected to FRED successfully
 Test series retrieved: 317 observations
 Date range: 1947-01-01 to 2026-01-01


## Economic Data Series
Downloaded 10 carefully selected economic series from FRED. These 
were not chosen randomly, each one was selected because decades of 
economic research has proven it contains meaningful information about 
where the US economy is heading.

## Data Dictionary — Understanding Every Variable

### Target Variables
*These are the three indicators MacroSense will predict*

| Variable | FRED Code | What It Measures | Unit | Why It Matters |
|---|---|---|---|---|
| **GDP** | GDPC1 | Real Gross Domestic Product — the total value of all goods and services produced in the US, adjusted for inflation | Billions of 2017 USD | The single most comprehensive measure of economic health. When GDP grows the economy is expanding. When it shrinks the economy is in trouble. |
| **CPI** | CPIAUCSL | Consumer Price Index — measures how much a standard basket of everyday goods and services costs over time | Index (1982-84 = 100) | The primary measure of inflation. When CPI rises too fast people's purchasing power falls. The Federal Reserve targets 2% annual CPI growth. |
| **Unemployment** | UNRATE | The percentage of working-age people who are actively looking for work but cannot find it | Percentage % | Measures labour market health. Rising unemployment signals economic distress. Falling unemployment signals a healthy growing economy. |

---

### Input Variables
*These are the indicators the model uses to predict the targets above*

| Variable | FRED Code | What It Measures | Unit | Why It Predicts The Future |
|---|---|---|---|---|
| **FedFunds** | FEDFUNDS | The Federal Funds Rate — the interest rate at which banks lend money to each other overnight, set by the Federal Reserve | Percentage % | When the Fed raises rates borrowing becomes expensive, spending slows, and GDP growth falls. Rate changes today affect the economy 6-18 months later — making this a powerful leading indicator. |
| **M2** | M2SL | Money Supply M2 — the total amount of money circulating in the US economy including cash, savings accounts, and money market funds | Billions of USD | More money in circulation tends to stimulate spending and growth. Rapid M2 growth often precedes inflation. Falling M2 growth signals economic slowdown ahead. |
| **YieldCurve** | T10Y2Y | The difference between 10-year and 2-year US Treasury bond interest rates | Percentage Points | This is arguably the most powerful recession predictor known. When it goes negative (called an inversion) a recession has followed within 6-18 months in every single case in US history since 1976. |
| **IndPro** | INDPRO | Industrial Production Index — measures how much factories, mines, and electric utilities are producing | Index (2017 = 100) | Factory output falls before recessions arrive because businesses sense slowing demand and cut production early. This makes it a reliable leading indicator. |
| **RetailSales** | RSAFS | Total monthly sales at retail and food service establishments across the US | Millions of USD | Consumer spending accounts for roughly 70% of US GDP. When retail sales fall GDP almost always follows. This is one of the most direct measures of economic momentum. |
| **Sentiment** | UMCSENT | University of Michigan Consumer Sentiment Index — a monthly survey measuring how optimistic US consumers feel about current and future economic conditions | Index | How people feel predicts how they spend. When sentiment falls consumers cut back on big purchases — cars, appliances, holidays — which directly reduces GDP growth. |
| **JoblessClaims** | ICSA | Initial Jobless Claims — the number of people filing for unemployment insurance benefits for the first time in a given week | Number of People | The most timely labour market indicator available (released weekly). Rising claims signal that companies are beginning to lay off workers — one of the earliest warning signs of economic deterioration. |

---

### Important Note on Data Coverage

Not all series go back equally far in time. Modern economic measurement 
developed gradually throughout the 20th century:

| Series | Approximate Start Date |
|---|---|
| CPI | 1913 |
| IndPro | 1919 |
| GDP | 1947 |
| Unemployment | 1948 |
| FedFunds | 1954 |
| Sentiment | 1952 |
| M2 | 1959 |
| JoblessClaims | 1967 |
| YieldCurve | 1976 |
| RetailSales | 1992 |

**This means our usable dataset effectively begins around 1992** , 
when all series are simultaneously available. We will handle this 
in the next notebook during data cleaning.

In [3]:
# Download all 10 economic series from FRED
# Each line pulls one specific data series
# The code in quotes is FRED's internal ID for each series


# TARGET VARIABLES — what we want to predict
gdp          = fred.get_series('GDPC1')      # Real GDP
cpi          = fred.get_series('CPIAUCSL')   # Inflation
unemployment = fred.get_series('UNRATE')     # Unemployment Rate

# INPUT VARIABLES — what helps us predict
fedfunds     = fred.get_series('FEDFUNDS')   # Interest Rate
m2           = fred.get_series('M2SL')       # Money Supply
yieldcurve   = fred.get_series('T10Y2Y')     # Yield Curve Spread
indpro       = fred.get_series('INDPRO')     # Industrial Production
retailsales  = fred.get_series('RSAFS')      # Retail Sales
sentiment    = fred.get_series('UMCSENT')    # Consumer Sentiment
claims       = fred.get_series('ICSA')       # Jobless Claims


print("\n All 10 series downloaded successfully!")


🎉 All 10 series downloaded successfully!


In [4]:
# Turn all 10 series into one master table
df = pd.DataFrame({
    'GDP'          : gdp,
    'CPI'          : cpi,
    'Unemployment' : unemployment,
    'FedFunds'     : fedfunds,
    'M2'           : m2,
    'YieldCurve'   : yieldcurve,
    'IndPro'       : indpro,
    'RetailSales'  : retailsales,
    'Sentiment'    : sentiment,
    'JoblessClaims': claims
})

# Resample everything to monthly frequency
df = df.resample('ME').last()

# Preview the table
print(f" Master table created successfully")
print(f"\n Table shape: {df.shape}")
print(f"   → {df.shape[0]} rows (months)")
print(f"   → {df.shape[1]} columns (variables)")
print(f"\n Date range: {df.index[0].date()} to {df.index[-1].date()}")
print(f"\n First 5 rows:")
df.head()

 Master table created successfully

 Table shape: (1289, 10)
   → 1289 rows (months)
   → 10 columns (variables)

 Date range: 1919-01-31 to 2026-05-31

 First 5 rows:


,GDP,CPI,Unemployment,FedFunds,M2,YieldCurve,IndPro,RetailSales,Sentiment,JoblessClaims
1919-01-31,NaN,NaN,NaN,NaN,NaN,NaN,4.8739,NaN,NaN,NaN
1919-02-28,NaN,NaN,NaN,NaN,NaN,NaN,4.6585,NaN,NaN,NaN
1919-03-31,NaN,NaN,NaN,NaN,NaN,NaN,4.5238,NaN,NaN,NaN
1919-04-30,NaN,NaN,NaN,NaN,NaN,NaN,4.6046,NaN,NaN,NaN
1919-05-31,NaN,NaN,NaN,NaN,NaN,NaN,4.6315,NaN,NaN,NaN


In [5]:
# Find out where Jupyter is currently working from
import os

current_location = os.getcwd()
print(f" Jupyter is currently working from:")
print(f"   {current_location}")

 Jupyter is currently working from:
   C:\Users\HP\Documents\MacroSense


In [6]:
# Save the master table as a CSV file

# Build the correct save path
save_path = os.path.join('data', 'raw_fred_data.csv')

# Save the table
df.to_csv(save_path)


## Verifying The Saved Data

We read the saved CSV file back into Python and run a final 
verification check to confirm:

1. The correct number of rows and columns were saved
2. All column names are present and correctly spelled
3. The date range is what we expected
4. We understand the missing value situation before moving forward

### Understanding Missing Values

Missing values in this dataset are **not errors.** They simply 
represent periods before certain measurements were invented or 
tracked. For example GDP was not measured before 1947 and retail 
sales were not tracked before 1992.

We will handle missing values properly in Notebook 2 by trimming 
the dataset to start from the point where all series have 
simultaneously available data.

In [7]:
# Read the saved file back and confirm everything is correct
df_check = pd.read_csv(save_path, index_col=0, parse_dates=True)

print(f"\n Shape: {df_check.shape}")
print(f"   {df_check.shape[0]} months of data")
print(f"    {df_check.shape[1]} economic variables")
print(f"\n First date: {df_check.index[0].date()}")
print(f" Last date:  {df_check.index[-1].date()}")
print(f"\n Column names:")
for col in df_check.columns:
    print(f"   → {col}")
print(f"\n Missing values per column:")
print(df_check.isnull().sum())


 Shape: (1289, 10)
   1289 months of data
    10 economic variables

 First date: 1919-01-31
 Last date:  2026-05-31

 Column names:
   → GDP
   → CPI
   → Unemployment
   → FedFunds
   → M2
   → YieldCurve
   → IndPro
   → RetailSales
   → Sentiment
   → JoblessClaims

 Missing values per column:
GDP              972
CPI              338
Unemployment     350
FedFunds         427
M2               482
YieldCurve       689
IndPro             1
RetailSales      877
Sentiment        618
JoblessClaims    576
dtype: int64


### Key Observations From This Notebook

1. **Data Volume:** 1,289 months of data spanning from 1919 to 
   May 2026 — over a century of economic history
   
2. **Missing Values:** Significant missing values exist in most 
   series because modern economic measurement developed gradually 
   throughout the 20th century. This is expected and will be 
   handled in Notebook 2.

3. **Effective Start Date:** Our usable dataset where all 10 
   variables are simultaneously available begins approximately 
   in **1992** — giving us roughly 30+ years of complete 
   multivariate data for modelling.

4. **Most Complete Series:** Industrial Production (IndPro) has 
   only 2 missing values — the most complete series in our dataset.

5. **Least Complete Series:** GDP has 972 missing values because 
   it was not measured before 1947 and is only reported quarterly.

---

### What Comes Next — Notebook 2: Exploratory Data Analysis

In the next notebook we will:
- Clean and trim the dataset to our usable date range
- Visualise every variable over time with recession shading
- Identify patterns and relationships between variables
- Produce the professional charts that will appear in our README
- Write our key findings and economic interpretations

**Proceed to:** `02_eda.ipynb`